# Random Sampling in Python 

This section summarizes the ways you have learned to sample at random using Python, and introduces a new way.

## Review: Sampling from a Population in a DataFrame 
If you are sampling from a population of individuals whose data are represented in the rows of a DataFrame, then you can use the DataFrame method `sample` to [randomly select rows](https://inferentialthinking.com/chapters/10/1/Empirical_Distributions.html#id1) of the DataFrame. That is, you can use `sample` to select a random sample of individuals.

`sample` draws uniformly at random. For rolls of a die, the call uses `with_replacement=True`. This is a natural model for chance experiments such as rolling a die.

In [1]:
import polars as pl
import hvplot.polars
path_data = '../../../assets/data/'
import sys
sys.path.append("..")
from sample_proportions_pl import sample_proportions_pl


In [2]:
faces = pl.int_range(1, 7, eager=True).alias("Face")
die = pl.DataFrame([faces])
die


Face
i64
1
2
3
4
5
6


Run the cell below to simulate 7 rolls of a die.

In [3]:
# POLARS_CHANGE: with_replacement=True matches Table.sample, which draws with replacement
die.sample(7, with_replacement=True)


Face
i64
2
5
1
3
6
5
4


Sometimes it is more natural to sample individuals at random without replacement. This is called a simple random sample. The argument `with_replacement=False` allows you to do this.

In [4]:
actors = pl.read_csv(path_data + 'actors.csv', eol_char='\r')
actors = actors.with_columns(
    pl.col("Total Gross").str.strip_chars().cast(pl.Float64),
    pl.col("Average per Movie").str.strip_chars().cast(pl.Float64),
    pl.col("Gross").str.strip_chars().cast(pl.Float64),
)
actors


Actor,Total Gross,Number of Movies,Average per Movie,#1 Movie,Gross
str,f64,i64,f64,str,f64
"""Harrison Ford""",4871.7,41,118.8,"""Star Wars: The Force Awakens""",936.7
"""Samuel L. Jackson""",4772.8,69,69.2,"""The Avengers""",623.4
"""Morgan Freeman""",4468.3,61,73.3,"""The Dark Knight""",534.9
"""Tom Hanks""",4340.8,44,98.7,"""Toy Story 3""",415.0
"""Robert Downey, Jr.""",3947.3,53,74.5,"""The Avengers""",623.4
…,…,…,…,…,…
"""Jeremy Renner""",2500.3,21,119.1,"""The Avengers""",623.4
"""Philip Seymour Hoffman""",2463.7,40,61.6,"""Catching Fire""",424.7
"""Sandra Bullock""",2462.6,35,70.4,"""Minions""",336.0


In [5]:
# Simple random sample of 5 rows
actors.sample(5, with_replacement=False)


Actor,Total Gross,Number of Movies,Average per Movie,#1 Movie,Gross
str,f64,i64,f64,str,f64
"""Jeremy Renner""",2500.3,21,119.1,"""The Avengers""",623.4
"""Bradley Cooper""",2557.7,25,102.3,"""American Sniper""",350.1
"""Anthony Daniels""",3162.9,7,451.8,"""Star Wars: The Force Awakens""",936.7
"""Bruce Willis""",3189.4,60,53.2,"""Sixth Sense""",293.5
"""Helena Bonham Carter""",2822.0,36,78.4,"""Harry Potter / Deathly Hallows…",381.0


Since `sample` gives you the entire sample in the order in which the rows were selected, you can use DataFrame methods on the sampled DataFrame to answer many questions about the sample. For example, you can find the number of times the die showed six spots, or the average number of movies in which the sampled actors appeared, or whether one two specified actors appeared in the sample. You might need multiple lines of code to get some of this information.

## Review: Sampling from a Population in a Series 

If you are sampling from a population of individuals whose data are represented as a Series, you can use `sample` to [randomly select elements of the Series](https://inferentialthinking.com/chapters/09/3/Simulation.html#example-number-of-heads-in-100-tosses).

A call with `with_replacement=True` samples at random with replacement.

In [6]:
# The faces of a die, as a Series
faces


Face
i64
1
2
3
4
5
6


In [7]:
# 7 rolls of the die
faces.sample(7, with_replacement=True)


Face
i64
2
4
2
4
2
2
1


The argument `with_replacement=False` allows you to get a simple random sample, that is, a sample drawn at random without replacement.

In [8]:
# Series of actor names
actor_names = actors.get_column("Actor")


In [9]:
# Simple random sample of 5 actor names
actor_names.sample(5, with_replacement=False)


Actor
str
"""Leonardo DiCaprio"""
"""Harrison Ford"""
"""Robin Williams"""
"""Ralph Fiennes"""
"""Robert Downey, Jr."""


Just as `sample` did, so also `Series.sample` gives you the entire sequence of sampled elements. You can use Series operations to answer many questions about the sample. For example, you can find which actor was the second one to be drawn, or the number of faces of the die that appeared more than once. Some answers might need multiple lines of code.

## Sampling from a Categorical Distribution 
Sometimes we are interested in a categorical attribute of our sampled individuals. For example, we might be looking at whether a coin lands Heads or Tails. Or we might be interested in the political parties of randomly selected voters.

In such cases, we frequently need the proportions of sampled voters in the different categories. If we have the entire sample, we can calculate these proportions. The function `sample_proportions_pl` does that work for us. It is tailored for sampling at random with replacement from a categorical distribution and returns the proportions of sampled elements in each category.

The `sample_proportions_pl` function takes two arguments:
- the sample size
- the distribution of the categories in the population, as a list of proportions that add up to 1

It returns a list containing the distribution of the categories in a random sample of the given size taken from the population. That's a list of the sample proportions in all the different categories, in the same order in which they appeared in the population distribution. The list is stored as a Series below.

For example, suppose each plant of a species is red-flowering with chance 25%, pink-flowering with chance 50%, and white-flowering with chance 25%, regardless of the flower colors of all other plants. You can use `sample_proportions_pl` to see the proportions of the different colors among 300 plants of the species.

In [10]:
# Species distribution of flower colors:
# Proportions are in the order Red, Pink, White
species_proportions = [0.25, 0.5, .25]

sample_size = 300

# Distribution of sample
# POLARS_CHANGE: sample_proportions_pl returns a list; the Series gives it .item
sample_distribution = pl.Series(
    "sample_distribution",
    sample_proportions_pl(sample_size, species_proportions),
)
sample_distribution


sample_distribution
f64
0.203333
0.476667
0.32


As you expect, the proportions in the sample sum to 1.

In [11]:
sum(sample_distribution)


1.0

The categories in `species_proportions` are in the order Red, Pink, White. That order is preserved by `sample_proportions_pl`. If you just want the proportion of pink-flowering plants in the sample, you can use `item`:

In [12]:
# Sample proportion of Heads
sample_distribution.item(1)


0.4766666666666667

You can use `sample_proportions_pl` and Series operations to answer questions based only on the proportions of sampled individuals in the different categories. You will not be able to answer questions that require more detailed information about the sample, such as which of the sampled plants had each of the different colors.